In [40]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [41]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [42]:

queryOderDetail = """
SELECT 
[SalesOrderID]
      ,[SalesOrderDetailID]
      ,[CarrierTrackingNumber]
      ,[OrderQty]
      ,[ProductID]
      ,[SpecialOfferID]
      ,[UnitPrice]
      ,[UnitPriceDiscount]
FROM Sales.SalesOrderDetail
"""

tablaSalesOrderDetail = pd.read_sql_query(queryOderDetail, motorBaseDatos)




queryOderHeader = """
SELECT 
[SalesOrderID]
      ,[RevisionNumber]
      ,[OrderDate]
      ,[DueDate]
      ,[ShipDate]
      ,[SalesOrderNumber]
      ,[CustomerID]
      ,[SalesPersonID]
      ,[TerritoryID]
      ,[TaxAmt]
      ,[Freight]
FROM Sales.SalesOrderHeader
"""

tablaSalesOrderHeader = pd.read_sql_query(queryOderHeader, motorBaseDatos)


queryDimensionProduct = """
SELECT 
      [ProductKey]
      ,[StandardCost]
FROM dbo.dimensionProduct
"""

dimensionProduct = pd.read_sql_query(queryDimensionProduct, motorBodegaDatos)


# tablaSalesOrderDetail
# tablaSalesOrderHeader
# dimensionProduct

TRANSFORMACION

In [43]:
tablaSales = tablaSalesOrderHeader.merge(tablaSalesOrderDetail, on='SalesOrderID')
tablaSales = tablaSales.merge(dimensionProduct, left_on='ProductID', right_on='ProductKey')


tablaSales

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,SalesOrderNumber,CustomerID,SalesPersonID,TerritoryID,TaxAmt,Freight,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,ProductKey,StandardCost
0,43659,8,2011-05-31,2011-06-12,2011-06-07,SO43659,29825,279.0,5,1971.5149,616.0984,1,4911-403C-98,1,776,1,2024.994,0.0,776,1898.0944
1,43659,8,2011-05-31,2011-06-12,2011-06-07,SO43659,29825,279.0,5,1971.5149,616.0984,2,4911-403C-98,3,777,1,2024.994,0.0,777,1898.0944
2,43659,8,2011-05-31,2011-06-12,2011-06-07,SO43659,29825,279.0,5,1971.5149,616.0984,3,4911-403C-98,1,778,1,2024.994,0.0,778,1898.0944
3,43659,8,2011-05-31,2011-06-12,2011-06-07,SO43659,29825,279.0,5,1971.5149,616.0984,4,4911-403C-98,1,771,1,2039.994,0.0,771,1912.1544
4,43659,8,2011-05-31,2011-06-12,2011-06-07,SO43659,29825,279.0,5,1971.5149,616.0984,5,4911-403C-98,1,772,1,2039.994,0.0,772,1912.1544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121312,75122,8,2014-06-30,2014-07-12,2014-07-07,SO75122,15868,NaN,6,2.4776,0.7743,121313,None,1,878,1,21.980,0.0,878,8.2205
121313,75122,8,2014-06-30,2014-07-12,2014-07-07,SO75122,15868,NaN,6,2.4776,0.7743,121314,None,1,712,1,8.990,0.0,712,6.9223
121314,75123,8,2014-06-30,2014-07-12,2014-07-07,SO75123,18759,NaN,6,15.1976,4.7493,121315,None,1,878,1,21.980,0.0,878,8.2205
121315,75123,8,2014-06-30,2014-07-12,2014-07-07,SO75123,18759,NaN,6,15.1976,4.7493,121316,None,1,879,1,159.000,0.0,879,59.4660


In [44]:

tablaSales.rename(columns={
    'TerritoryID': 'SalesTerritoryKey',
    'CustomerID': 'CustomerKey',
    'UnitPriceDiscount': 'UnitPriceDiscountPct',
    'SpecialOfferID': 'PromotionKey',
    # 'ProductID': 'ProductKey',
    'OrderQty': 'OrderQuantity',
    'StandardCost': 'ProductStandardCost',
}, inplace=True)

tablaSales["ExtendedAmount"] = tablaSales["UnitPrice"]
tablaSales["DiscountAmount"] = 0
tablaSales["TotalProductCost"] = tablaSales["ProductStandardCost"]
tablaSales["SalesAmount"] = None
tablaSales["CustomerPONumber"] = None

tablaSales["OrderDateKey"] = pd.to_datetime(tablaSales["OrderDate"]).dt.strftime('%Y%m%d')
tablaSales["DueDateKey"] = pd.to_datetime(tablaSales["DueDate"]).dt.strftime('%Y%m%d')
tablaSales["ShipDateKey"] = pd.to_datetime(tablaSales["ShipDate"]).dt.strftime('%Y%m%d')

tablaSales.drop(columns=[
    'SalesOrderID',
    'SalesPersonID'

], inplace=True)

# hechoInternetSales = tablaSales

CARGAR A LA BODEGA

In [ ]:
tablaSales.to_sql('hechoInternetSales',motorBodegaDatos, if_exists='replace',index=False)